# Klasifikasi Kematangan Tapai - Linear Regression

Notebook ini menggunakan **Logistic Regression** (regresi linear untuk klasifikasi) dari scikit-learn untuk memprediksi status kematangan tapai berdasarkan data sensor (suhu, kelembaban, kadar gas) selama waktu fermentasi.

## Dataset
- **Fitur:** `jam`, `suhu`, `kelembaban`, `kadar_gas`
- **Target:** `status_kematangan` (belum matang, matang, terlalu matang)
- **10 percobaan**, masing-masing 60 jam pengamatan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('seaborn-v0_8')
print('Libraries loaded successfully')

## 1. Load & Eksplorasi Data

In [ ]:
df = pd.read_csv('dataset/dataset_kematangan_tapai_v2.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
print('Info dataset:')
df.info()
print('\nDistribusi label:')
print(df['status_kematangan'].value_counts())

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'belum matang': '#3498db', 'matang': '#2ecc71', 'terlalu matang': '#e74c3c'}

for ax, col in zip(axes, ['suhu', 'kelembaban', 'kadar_gas']):
    for label, color in colors.items():
        subset = df[df['status_kematangan'] == label]
        ax.scatter(subset['jam'], subset[col], c=color, label=label, alpha=0.5, s=20)
    ax.set_xlabel('Jam')
    ax.set_ylabel(col.capitalize())
    ax.set_title(f'{col.capitalize()} vs Jam')
    ax.legend()

plt.suptitle('Distribusi Fitur Berdasarkan Status Kematangan', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_linear_regression_eda.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Preprocessing

In [ ]:
# Encode label
label_map = {'belum matang': 0, 'matang': 1, 'terlalu matang': 2}
df['label'] = df['status_kematangan'].map(label_map)

# Fitur dan target
X = df[['jam', 'suhu', 'kelembaban', 'kadar_gas']].values
y = df['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisasi
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Training Model - Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

## 4. Evaluasi Model

In [ ]:
target_names = ['belum matang', 'matang', 'terlalu matang']
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix - Logistic Regression\nAccuracy: {accuracy*100:.2f}%')
plt.tight_layout()
plt.savefig('plot_linear_regression_cm.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Visualisasi Koefisien

In [ ]:
feature_names = ['jam', 'suhu', 'kelembaban', 'kadar_gas']
coef_df = pd.DataFrame(model.coef_, columns=feature_names, index=target_names)

fig, ax = plt.subplots(figsize=(9, 4))
coef_df.T.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_xlabel('Fitur')
ax.set_ylabel('Koefisien')
ax.set_title('Koefisien Logistic Regression per Kelas')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(title='Status')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('plot_linear_regression_coef.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Kesimpulan

| Metrik | Nilai |
|--------|-------|
| Model | Logistic Regression (Linear) |
| Fitur | jam, suhu, kelembaban, kadar_gas |
| Split | 80% train / 20% test |

**Catatan:** Logistic Regression merupakan model linear untuk klasifikasi. Model ini mengasumsikan batas keputusan yang linear antar kelas dan **tidak mempertimbangkan urutan waktu** (data tiap jam dianggap independen). Hal ini menjadi keterbatasan utama karena data kematangan tapai sangat bergantung pada runtutan waktu fermentasi.